# Deutsche Bahn Job Scraper
Scrapes db.jobs, scores against CV, writes to SQLite.

**Run order:** Execute cells top to bottom. Sections 3 and 5 make network calls — don't re-run accidentally.

## 0. Imports & setup

In [2]:
import sys, os
import requests
from bs4 import BeautifulSoup
import re
import time
import json
import sqlite3
import urllib.parse
from datetime import date
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append(os.getcwd())
from config_isis import (
    CV_TEXT, HIGH_WEIGHT_TERMS, MEDIUM_WEIGHT_TERMS,
    IGNORE_TERMS, ALLOWED_LOCATIONS, MIN_SCORE, KEYWORD_BONUS_CAP
)

print('Imports OK')
print(f'Config loaded — {len(HIGH_WEIGHT_TERMS)} high-weight terms, {len(IGNORE_TERMS)} ignore terms')

Imports OK
Config loaded — 21 high-weight terms, 27 ignore terms


## 1. CV profile & scoring config
Loaded from `config_isis.py`. Nothing to edit here — run to verify what was imported.

In [3]:
print('=== CV TEXT (first 300 chars) ===')
print(CV_TEXT[:300])
print()
print('=== HIGH WEIGHT TERMS ===')
print(HIGH_WEIGHT_TERMS)
print()
print('=== ALLOWED LOCATIONS ===')
print(ALLOWED_LOCATIONS)

=== CV TEXT (first 300 chars) ===

Senior Analytics und Insights Führungskraft mit 10 Jahren Erfahrung im E-Commerce.
SQL Experte BigQuery Datenanalyse Business Intelligence KPI Kennzahlen Reporting.
Kundenfeedback Kundeninsights Voice of Customer VoC NPS Retourenanalyse Funnel-Analyse.
Machine Learning ML Python scikit-learn NLP Te

=== HIGH WEIGHT TERMS ===
['sql', 'datenanalyse', 'data analysis', 'analytics', 'business intelligence', 'bi', 'looker', 'product analytics', 'kundeninsights', 'customer insights', 'kpi', 'dashboard', 'reporting', 'e-commerce', 'ecommerce', 'digital', 'retouren', 'merchandising', 'digitalisierung', 'daten', 'data']

=== ALLOWED LOCATIONS ===
['berlin', 'wien', 'vienna', 'remote', 'wo du willst', 'home office', 'homeoffice']


## 2. Scraper config (DB-specific)
**This is the only section to update when adjusting DB scraping behaviour.**

Key difference from Siemens Energy: DB uses keyword-based search queries instead of offset pagination.

In [4]:
# ── Identity ──────────────────────────────────────────────────────────────────
SOURCE = 'deutsche_bahn'
DB_PATH = 'data/jobs.db'

# ── Search config ─────────────────────────────────────────────────────────────
BASE_URL = 'https://db.jobs'
SEARCH_URL = 'https://db.jobs/service/search/de-de/5441588'

SEARCH_QUERIES = [
    'Analyst',
    'Analytics',
    'Data',
    'Digitalisierung',
    'Product Manager',
    'Produktmanager',
    'Business Intelligence',
    'Reporting',
]

MAX_PAGES_PER_QUERY = 5   # 5 pages x 20 jobs x 8 queries = up to 800 before dedup

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'de-DE,de;q=0.9,en;q=0.8',
}

# DB uses 'Wo Du Willst' for remote — different from Siemens Energy
DB_ALLOWED_LOCATIONS = ['berlin', 'wo du willst']

SLEEP_BETWEEN_PAGES = 1.5
SLEEP_BETWEEN_DETAILS = 1.0

# Company-specific keyword overrides — merged with global config
EXTRA_IGNORE = [
    'ingenieur', 'leit- und sicherungstechnik', 'trainee',
]
EXTRA_HIGH_WEIGHT = []
EXTRA_MEDIUM_WEIGHT = []

IGNORE_TERMS_FINAL = IGNORE_TERMS + EXTRA_IGNORE
HIGH_WEIGHT_FINAL = HIGH_WEIGHT_TERMS + EXTRA_HIGH_WEIGHT
MEDIUM_WEIGHT_FINAL = MEDIUM_WEIGHT_TERMS + EXTRA_MEDIUM_WEIGHT

print(f'Source: {SOURCE}')
print(f'Queries: {len(SEARCH_QUERIES)} x {MAX_PAGES_PER_QUERY} pages = up to {len(SEARCH_QUERIES) * MAX_PAGES_PER_QUERY * 20} jobs before dedup')
print(f'Location filter: {DB_ALLOWED_LOCATIONS}')
print(f'Ignore terms (total): {len(IGNORE_TERMS_FINAL)}')

Source: deutsche_bahn
Queries: 8 x 5 pages = up to 800 jobs before dedup
Location filter: ['berlin', 'wo du willst']
Ignore terms (total): 30


## 3. Scrape listing pages
Runs all search queries, filters by location at scrape time.

⚠️ **Network calls happen here.** Don't re-run unless you want to re-scrape.

In [5]:
def is_allowed_location(location):
    loc_lower = location.lower()
    return any(allowed in loc_lower for allowed in DB_ALLOWED_LOCATIONS)


def fetch_listing_page(query, page_num):
    """Fetch one page of search results for a given query."""
    params = {
        'country': 'Deutschland',
        'qli': 'true',
        'query': query,
        'targetGroup': '',
        'sort': 'score',
        'page': page_num,
    }
    url = SEARCH_URL + '?' + urllib.parse.urlencode(params)

    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f'  ⚠️  Query "{query}" page {page_num} failed: {e}')
        return []

    soup = BeautifulSoup(resp.text, 'html.parser')
    jobs = []

    for hit in soup.select('.m-search-hit'):
        title_el = hit.select_one('.m-search-hit__title')
        title = title_el.get_text(strip=True) if title_el else ''

        link = hit.get('href', '')
        if link and not link.startswith('http'):
            link = BASE_URL + link

        job_id = hit.get('data-job-id', '')

        detail_items = hit.select('.m-search-hit__detail-item, li')
        details = [d.get_text(strip=True) for d in detail_items if d.get_text(strip=True)]
        location = details[0] if len(details) > 0 else ''
        company = details[1] if len(details) > 1 else 'Deutsche Bahn'
        start_date = details[2] if len(details) > 2 else ''

        if title and is_allowed_location(location):
            jobs.append({
                'source': SOURCE,
                'source_job_id': job_id,
                'title': title,
                'company': company,
                'location': location,
                'start_date': start_date,
                'link': link,
                'description': '',
            })

    return jobs


# ── Run scrape ────────────────────────────────────────────────────────────────
raw_jobs = []
print(f'Scraping DB jobs | {len(SEARCH_QUERIES)} queries x {MAX_PAGES_PER_QUERY} pages...')

for query in SEARCH_QUERIES:
    print(f"\n  Query: '{query}'")
    for page in range(1, MAX_PAGES_PER_QUERY + 1):
        jobs = fetch_listing_page(query, page)
        if not jobs:
            print(f'    Page {page}: no results, stopping query')
            break
        raw_jobs.extend(jobs)
        print(f'    Page {page}: {len(jobs)} location-matched jobs (running total: {len(raw_jobs)})')
        time.sleep(SLEEP_BETWEEN_PAGES)

# Deduplicate by source_job_id
seen = set()
unique_jobs = []
for j in raw_jobs:
    if j['source_job_id'] not in seen:
        seen.add(j['source_job_id'])
        unique_jobs.append(j)

print(f'\n✅ {len(unique_jobs)} unique location-matched jobs scraped')

Scraping DB jobs | 8 queries x 5 pages...

  Query: 'Analyst'
    Page 1: 1 location-matched jobs (running total: 1)
    Page 2: 1 location-matched jobs (running total: 2)
    Page 3: 1 location-matched jobs (running total: 3)
    Page 4: 1 location-matched jobs (running total: 4)
    Page 5: 1 location-matched jobs (running total: 5)

  Query: 'Analytics'
    Page 1: 5 location-matched jobs (running total: 10)
    Page 2: 5 location-matched jobs (running total: 15)
    Page 3: 5 location-matched jobs (running total: 20)
    Page 4: 5 location-matched jobs (running total: 25)
    Page 5: 5 location-matched jobs (running total: 30)

  Query: 'Data'
    Page 1: 6 location-matched jobs (running total: 36)
    Page 2: 6 location-matched jobs (running total: 42)
    Page 3: 6 location-matched jobs (running total: 48)
    Page 4: 6 location-matched jobs (running total: 54)
    Page 5: 6 location-matched jobs (running total: 60)

  Query: 'Digitalisierung'
    Page 1: 6 location-matched jobs 

In [6]:
# Inspect raw results
df_raw = pd.DataFrame(unique_jobs)
print(f'Shape: {df_raw.shape}')
df_raw[['title', 'source_job_id', 'location']].head(10)

Shape: (14, 8)


,title,source_job_id,location
0,IT Business Analyst:in Payroll,620525,Wo Du Willst-Job
1,BI Engineer:in TM1 / Planning Analytics,616270,"Berlin, Deutschland"
2,Werkstudent:in Controlling & Business Intellig...,621149,Wo Du Willst-Job
3,Werkstudent:in Business Analytics & Reporting,618252,Wo Du Willst-Job
4,Werkstudent:in Prozessanalyse und Digitalisierung,620374,"Berlin, Halle (Saale), Deutschland"
5,Praktikum Data Science & -management Nachhalti...,620053,Wo Du Willst-Job
6,Schweißfachingenieur:in als Qualitätsprüfingen...,351022,"Berlin, Deutschland"
7,Senior Backend Entwickler:in Geospatial Solutions,619309,Wo Du Willst-Job
8,Senior Manager Asset Management Rolling Stock ...,619948,"Berlin, Deutschland"
9,Technische:r IT-Projektleiter:in,621212,Wo Du Willst-Job


## 4. Filter
Drops irrelevant jobs by title before fetching descriptions.

Check the dropped list — if real jobs are being filtered, update EXTRA_IGNORE in Section 2 and re-run this cell only (no re-scraping needed).

In [7]:
def is_irrelevant(title):
    title_lower = title.lower()
    return any(term in title_lower for term in IGNORE_TERMS_FINAL)

before = len(unique_jobs)
filtered_jobs = [j for j in unique_jobs if not is_irrelevant(j['title'])]
dropped = [j for j in unique_jobs if is_irrelevant(j['title'])]

print(f'Before filter: {before}')
print(f'After filter:  {len(filtered_jobs)}')
print(f'Dropped:       {len(dropped)}')
print()
print('--- Dropped titles (sanity check) ---')
for j in dropped:
    print(f"  {j['title']}")

Before filter: 14
After filter:  5
Dropped:       9

--- Dropped titles (sanity check) ---
  Werkstudent:in Controlling & Business Intelligence
  Werkstudent:in Business Analytics & Reporting
  Werkstudent:in Prozessanalyse und Digitalisierung
  Praktikum Data Science & -management Nachhaltigkeit und Umwelt 2026 (w/m/d)
  Schweißfachingenieur:in als Qualitätsprüfingenieur:in für die Regionen Polen und Tschechien
  Werkstudent:in Engineering Planen & Bauen
  Trainee Digitale Schiene - Projektmanagement Stellwerksprojekte (w/m/d)
  Trainee Digitale Schiene - Projektmanagement Hochbau (w/m/d)
  Ingenieur:in als Projektleiter:in S-Bahnstromversorgung


## 5. Fetch descriptions
Fetches each job detail page and extracts full JD text from DB's embedded JS object.

⚠️ **Slow cell.** Checkpoint JSON saved after completion — reload from it if kernel restarts.

In [8]:
def fetch_description(job):
    """
    Fetches DB job detail page.
    Description is embedded in a JS digitalData object as jobTasks + jobProfile.
    """
    if not job.get('link'):
        return job['title']

    try:
        resp = requests.get(job['link'], headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"    ⚠️  Failed: {job['title'][:40]}: {e}")
        return job['title']

    html = resp.text
    tasks_match = re.search(r'"jobTasks":\s*"(.*?)",\s*"jobProfile"', html, re.DOTALL)
    profile_match = re.search(r'"jobProfile":\s*"(.*?)",\s*"language"', html, re.DOTALL)

    tasks = tasks_match.group(1) if tasks_match else ''
    profile = profile_match.group(1) if profile_match else ''

    combined = (tasks + ' ' + profile).replace('\\/', '/').replace('\\n', ' ').replace('\\"', '"')
    clean = re.sub(r'<[^>]+>', ' ', combined)
    clean = re.sub(r'\s+', ' ', clean).strip()

    return clean if clean else job['title']


enriched_jobs = []
print(f'Fetching descriptions for {len(filtered_jobs)} jobs...')

for i, job in enumerate(filtered_jobs, 1):
    print(f"  [{i}/{len(filtered_jobs)}] {job['title'][:50]}...", end=' ')
    desc = fetch_description(job)
    job['description'] = desc
    enriched_jobs.append(job)
    print(f'{len(desc.split())} words')
    time.sleep(SLEEP_BETWEEN_DETAILS)

checkpoint_path = f'data/{SOURCE}_checkpoint.json'
with open(checkpoint_path, 'w', encoding='utf-8') as f:
    json.dump(enriched_jobs, f, ensure_ascii=False, indent=2)
print(f'\n✅ Descriptions fetched. Checkpoint saved to {checkpoint_path}')

Fetching descriptions for 5 jobs...
  [1/5] IT Business Analyst:in Payroll... 395 words
  [2/5] BI Engineer:in TM1 / Planning Analytics... 314 words
  [3/5] Senior Backend Entwickler:in Geospatial Solutions... 279 words
  [4/5] Senior Manager Asset Management Rolling Stock und ... 347 words
  [5/5] Technische:r IT-Projektleiter:in... 307 words

✅ Descriptions fetched. Checkpoint saved to data/deutsche_bahn_checkpoint.json


In [ ]:
# Reload from checkpoint if kernel restarted after Section 5:
# with open(f'data/{SOURCE}_checkpoint.json', encoding='utf-8') as f:
#     enriched_jobs = json.load(f)
# print(f'Loaded {len(enriched_jobs)} jobs from checkpoint')

## 6. Location filter
DB pre-filters location in Section 3, so this is mainly a safety check.

In [9]:
before = len(enriched_jobs)
location_filtered = [j for j in enriched_jobs if is_allowed_location(j['location'])]
dropped_location = [j for j in enriched_jobs if not is_allowed_location(j['location'])]

print(f'Before: {before}  |  After: {len(location_filtered)}  |  Dropped: {len(dropped_location)}')
if dropped_location:
    for j in dropped_location:
        print(f"  {j['title'][:50]} | {j['location']}")
else:
    print('No jobs dropped — location was pre-filtered in Section 3 (expected)')

Before: 5  |  After: 5  |  Dropped: 0
No jobs dropped — location was pre-filtered in Section 3 (expected)


## 7. Score
TF-IDF cosine similarity + keyword bonus. Review top 20 before writing to DB.

In [10]:
def keyword_bonus(text):
    text_lower = text.lower()
    bonus = 0
    for term in HIGH_WEIGHT_FINAL:
        if term in text_lower:
            bonus += 8
    for term in MEDIUM_WEIGHT_FINAL:
        if term in text_lower:
            bonus += 3
    return min(bonus, KEYWORD_BONUS_CAP)


def score_jobs(jobs):
    if not jobs:
        print('No jobs to score.')
        return jobs
    descriptions = [j['description'] for j in jobs]
    corpus = [CV_TEXT] + descriptions
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(corpus)
    similarities = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1:])[0]
    for i, job in enumerate(jobs):
        base = round(float(similarities[i]) * 100, 1)
        bonus = keyword_bonus(job['description'])
        job['score'] = min(round(base + bonus, 1), 100)
    return sorted(jobs, key=lambda x: x['score'], reverse=True)


scored_jobs = score_jobs(location_filtered)
print(f'✅ Scored {len(scored_jobs)} jobs')

✅ Scored 5 jobs


In [11]:
df_scored = pd.DataFrame(scored_jobs)[['score', 'title', 'location', 'link']]
df_scored.head(20)

,score,title,location,link
0,35.4,BI Engineer:in TM1 / Planning Analytics,"Berlin, Deutschland",https://db.jobs/de-de/Suche/BI-Engineer-in-TM1...
1,33.3,IT Business Analyst:in Payroll,Wo Du Willst-Job,https://db.jobs/de-de/Suche/IT-Business-Analys...
2,32.7,Technische:r IT-Projektleiter:in,Wo Du Willst-Job,https://db.jobs/de-de/Suche/Technische-r-IT-Pr...
3,31.0,Senior Manager Asset Management Rolling Stock ...,"Berlin, Deutschland",https://db.jobs/de-de/Suche/Senior-Manager-Ass...
4,28.9,Senior Backend Entwickler:in Geospatial Solutions,Wo Du Willst-Job,https://db.jobs/de-de/Suche/Senior-Backend-Ent...


## 8. Write to SQLite
Upserts into `jobs` table. Safe to re-run — won't create duplicates.

⚠️ **Only run when happy with scores above.**

In [ ]:
# Requires jobs table to exist — run "python setup_db.py" once from repo root in terminal if not already done. should say "DB created"

In [14]:
def write_to_db(jobs, db_path, min_score):
    today = str(date.today())
    to_write = [j for j in jobs if j['score'] >= min_score]
    skipped = len(jobs) - len(to_write)
    conn = sqlite3.connect(db_path)
    try:
        conn.executemany("""
            INSERT OR REPLACE INTO jobs
                (run_date, source, source_job_id, company, title, location,
                 start_date, link, score, description)
            VALUES
                (:run_date, :source, :source_job_id, :company, :title, :location,
                 :start_date, :link, :score, :description)
        """, [{**j, 'run_date': today} for j in to_write])
        conn.commit()
        print(f'✅ Wrote {len(to_write)} jobs to {db_path}')
        print(f'   Skipped {skipped} jobs below MIN_SCORE ({min_score})')
    except Exception as e:
        conn.rollback()
        print(f'❌ DB write failed: {e}')
        raise
    finally:
        conn.close()

write_to_db(scored_jobs, DB_PATH, MIN_SCORE)

✅ Wrote 5 jobs to data/jobs.db
   Skipped 0 jobs below MIN_SCORE (15)


In [15]:
# Verify
conn = sqlite3.connect(DB_PATH)
df_db = pd.read_sql(
    f"SELECT run_date, source, title, location, score FROM jobs WHERE source = '{SOURCE}' ORDER BY score DESC",
    conn
)
conn.close()
print(f'{len(df_db)} rows in DB for source={SOURCE}')
df_db.head(10)

5 rows in DB for source=deutsche_bahn


,run_date,source,title,location,score
0,2026-04-20,deutsche_bahn,BI Engineer:in TM1 / Planning Analytics,"Berlin, Deutschland",35.4
1,2026-04-20,deutsche_bahn,IT Business Analyst:in Payroll,Wo Du Willst-Job,33.3
2,2026-04-20,deutsche_bahn,Technische:r IT-Projektleiter:in,Wo Du Willst-Job,32.7
3,2026-04-20,deutsche_bahn,Senior Manager Asset Management Rolling Stock ...,"Berlin, Deutschland",31.0
4,2026-04-20,deutsche_bahn,Senior Backend Entwickler:in Geospatial Solutions,Wo Du Willst-Job,28.9
